# Notebook 03 — Silver Stream Processing
This notebook reads from Bronze Delta Lake as a stream validates every record, computes fraud signals,
normalises currencies to EUR, and writes clean data to Silver Delta Lake Bad records go to quarantine — never silently dropped
### Flow:
####   Bronze Delta Lake (stream)
         ↓ Spark Structured Streaming
         ↓ validate 6 rules
         ↓ cast types
         ↓ compute 5 fraud signals
         ↓ normalise to EUR
   ┌─────┴──────┐
   ↓            ↓
   
 Silver      Quarantine
 Delta Lake  bucket


In [1]:
# Install dependencies
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "delta-spark==3.1.0", "python-dotenv", "pymongo",],check=True)
print("Dependencies installed sucessfully")

Dependencies installed sucessfully


In [2]:
import os 
import uuid
import json
import logging
from datetime import datetime, date
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType, BooleanType, IntegerType

load_dotenv(find_dotenv(), override=True) # load secrets from env file


CONFIG = {
    #  ============== MinIO connection — same as Bronze notebook ==================================
    "minio_endpoint":    os.getenv("MINIO_ENDPOINT",    "http://minio:9000"),
    "minio_access_key":  os.getenv("MINIO_ACCESS_KEY",  ""),
    "minio_secret_key":  os.getenv("MINIO_SECRET_KEY",  ""),

    # ============== Storage paths — where to read and write ==================================
    "bronze_bucket":     os.getenv("BRONZE_BUCKET",     "bronze"),
    "silver_bucket":     os.getenv("SILVER_BUCKET",     "silver"),
    "quarantine_bucket": os.getenv("QUARANTINE_BUCKET", "quarantine"),

    # ============== MongoDB — audit log
    "mongodb_uri":       os.getenv("MONGODB_URI",       ""),
    "mongodb_db":        os.getenv("MONGODB_DB",        "atlas_pipeline"),

    # ============== Pipeline identity
    "pipeline_name":     "atlas-stream-silver",
    "processing_date":   str(date.today()),

    # Streaming settings Silver processes every 30 seconds — more work per record
    # than Bronze, so give it more time between batches.
    "trigger_interval":  "30 seconds",

    # ============== FX rates — PLN is the base currency in Poland ==============
    # All amounts normalised to EUR for cross-bank comparison
    "fx_rates_to_eur": {
        "PLN": 0.23,   # 1 PLN = 0.23 EUR
        "USD": 0.92,   # 1 USD = 0.92 EUR
        "GBP": 1.17,   # 1 GBP = 1.17 EUR
        "AUD": 0.60,   # 1 AUD = 0.60 EUR
        "SGD": 0.70,   # 1 SGD = 0.70 EUR
        "EUR": 1.00,   # already EUR — no conversion
    },

    # ── Fraud detection thresholds ────────────────────────────
    # Centralised here so data science team can tune them
    # without touching transformation code
    "fraud_large_amount_eur":   5000.0,   # above this = suspicious
    "fraud_ip_ranges":          [          # known Tor exit nodes
        "185.220.101",
        "198.96.155",
        "23.129.64",
        "185.220.100",
    ],
    "fraud_unknown_category":   "unknown", # merchant category flag
    "valid_currencies":         ["PLN", "EUR", "USD", "GBP", "AUD", "SGD"],
    "valid_bank_codes":         ["PKO", "MBANK", "ING", "ALIOR", "SANTANDER"],
}

# Generate unique run ID for this Silver streaming session
RUN_ID = str(uuid.uuid4())

print(f"Pipeline:         {CONFIG['pipeline_name']}")
print(f"Run ID:           {RUN_ID}")
print(f"Processing date:  {CONFIG['processing_date']}")
print(f"Trigger interval: {CONFIG['trigger_interval']}")
print(f"Valid currencies: {CONFIG['valid_currencies']}")
print(f"Valid banks:      {CONFIG['valid_bank_codes']}")
print(f"Fraud threshold:  EUR {CONFIG['fraud_large_amount_eur']:,.0f}")
print()
print("FX rates to EUR:")
for ccy, rate in CONFIG["fx_rates_to_eur"].items():
    print(f"  {ccy} × {rate} → EUR")
print()
print("Configuration loaded successfully")

Pipeline:         atlas-stream-silver
Run ID:           bcd56a73-173d-4a2f-9d1f-437ce2c3be02
Processing date:  2026-04-09
Trigger interval: 30 seconds
Valid currencies: ['PLN', 'EUR', 'USD', 'GBP', 'AUD', 'SGD']
Valid banks:      ['PKO', 'MBANK', 'ING', 'ALIOR', 'SANTANDER']
Fraud threshold:  EUR 5,000

FX rates to EUR:
  PLN × 0.23 → EUR
  USD × 0.92 → EUR
  GBP × 1.17 → EUR
  AUD × 0.6 → EUR
  SGD × 0.7 → EUR
  EUR × 1.0 → EUR

Configuration loaded successfully


In [3]:
# ─────────────────────────────────────────────────────────────
# Cell 3 — Create Spark Session
#
# Identical to Bronze notebook Cell 3
# Silver uses the same Spark configuration
# The only difference is the app name
# so we can distinguish Bronze and Silver
# in the Spark UI at http://localhost:4040
#
# No Kafka packages needed here
# Silver reads from Delta Lake not Kafka
# Delta Lake support is already included
# in the delta-spark package
# ─────────────────────────────────────────────────────────────

import os
from pyspark.sql import SparkSession

# Set Maven packages — Delta Lake and S3A only
# No Kafka connector needed for Silver
# Silver reads Bronze Delta Lake not Kafka topics
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.1.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.367 "
    "pyspark-shell"
)

# Stop any existing session cleanly
# Important — Spark only allows one active session
# If Bronze notebook is still running stop it first
try:
    SparkSession.getActiveSession().stop()
    import time
    time.sleep(2)
    print("Stopped existing Spark session")
except:
    pass

spark = (
    SparkSession.builder
    .appName("Atlas — Stream Silver Processing")

    # local[2] — 2 CPU cores inside Jupyter container
    # Same as Bronze — Silver runs in same environment
    .master("local[2]")

    # ── Delta Lake ────────────────────────────────────────────
    # Required to read and write Delta Lake format
    # Enables ACID transactions and time travel
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    # ── Enable Delta Lake streaming source ────────────────────
    # This tells Spark that Delta Lake tables can be
    # used as streaming sources — not just batch sources
    # Without this readStream.format("delta") fails
    .config(
        "spark.delta.streaming.enableDeltaLog",
        "true"
    )

    # ── Performance ───────────────────────────────────────────
    # 4 shuffle partitions for local 2-core development
    .config("spark.sql.shuffle.partitions", "4")

    # ── MinIO / S3A ───────────────────────────────────────────
    # Identical to Bronze — same MinIO connection
    # Silver reads from bronze bucket and writes to silver bucket
    # both on the same MinIO instance
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        CONFIG["minio_endpoint"]
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        CONFIG["minio_access_key"]
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        CONFIG["minio_secret_key"]
    )
    # No SSL — local MinIO has no certificate
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    # Path style URLs — required for MinIO
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    # S3A filesystem implementation class
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    # Credentials provider — uses access/secret key pair
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    # Maximum parallel connections to MinIO
    .config(
        "spark.hadoop.fs.s3a.connection.maximum",
        "100"
    )

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version:    {spark.version}")
print(f"Spark master:     {spark.sparkContext.master}")
print(f"App name:         {spark.sparkContext.appName}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Spark UI:         http://localhost:4040")
print()
print("Spark session ready for Silver processing")
print("No Kafka packages — Silver reads Bronze Delta Lake directly")

Spark version:    3.5.0
Spark master:     local[2]
App name:         Atlas — Stream Silver Processing
Shuffle partitions: 4
Spark UI:         http://localhost:4040

Spark session ready for Silver processing
No Kafka packages — Silver reads Bronze Delta Lake directly


In [4]:
# ─────────────────────────────────────────────────────────────
# Cell 5 — Silver transformation functions (fixed)
# pipe() helper defined FIRST before it is used
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# ─────────────────────────────────────────────────────────────
# HELPER — pipe() must be defined first
# Enables clean function chaining on DataFrames
# df.pipe(func1).pipe(func2) instead of func2(func1(df))
# ─────────────────────────────────────────────────────────────
def pipe(df, func):
    return func(df)

# ─────────────────────────────────────────────────────────────
# STAGE 1 — VALIDATION
# Add validation_error column — null means valid
# ─────────────────────────────────────────────────────────────
def add_validation(df):
    return df.withColumn(
        "validation_error",
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            F.lit("missing_transaction_id")
        ).when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            F.lit("missing_customer_id")
        ).when(
            F.col("amount").isNull() |
            (F.trim(F.col("amount")) == ""),
            F.lit("missing_amount")
        ).when(
            F.col("amount").cast(DecimalType(15, 2)).isNull() |
            (F.col("amount").cast(DecimalType(15, 2)) <= 0),
            F.lit("invalid_amount")
        ).when(
            ~F.upper(F.trim(F.col("currency")))
             .isin(CONFIG["valid_currencies"]),
            F.lit("invalid_currency")
        ).when(
            ~F.upper(F.trim(F.col("bank_code")))
             .isin(CONFIG["valid_bank_codes"]),
            F.lit("unknown_bank_code")
        ).when(
            F.to_timestamp(F.col("timestamp")).isNull(),
            F.lit("invalid_timestamp")
        ).otherwise(F.lit(None).cast("string"))
    )

print("add_validation defined — 7 rules")

# ─────────────────────────────────────────────────────────────
# STAGE 2 — TYPE CASTING
# ─────────────────────────────────────────────────────────────
def cast_types(df):
    return (df
        .withColumn("amount",
            F.col("amount").cast(DecimalType(15, 2)))
        .withColumn("event_timestamp",
            F.to_timestamp(F.col("timestamp")))
        .withColumn("currency",
            F.upper(F.trim(F.col("currency"))))
        .withColumn("bank_code",
            F.upper(F.trim(F.col("bank_code"))))
        .withColumn("status",
            F.upper(F.trim(F.col("status"))))
        .withColumn("merchant_category",
            F.lower(F.trim(F.col("merchant_category"))))
        .drop("timestamp")
    )

print("cast_types defined")

# ─────────────────────────────────────────────────────────────
# STAGE 3 — FRAUD SIGNALS
# ─────────────────────────────────────────────────────────────
def add_fraud_signals(df):
    ip_fraud_expr = F.lit(False)
    for prefix in CONFIG["fraud_ip_ranges"]:
        ip_fraud_expr = ip_fraud_expr | \
            F.col("ip_address").startswith(prefix)

    return (df
        .withColumn("is_large_amount",
            F.col("amount") > CONFIG["fraud_large_amount_eur"])
        .withColumn("is_suspicious_ip", ip_fraud_expr)
        .withColumn("is_unknown_merchant",
            F.col("merchant_category") ==
            CONFIG["fraud_unknown_category"])
        .withColumn("is_pending_status",
            F.col("status") == "PENDING")
        .withColumn("is_unusual_currency",
            ~F.col("currency").isin(["PLN","EUR","USD","GBP"]))
        .withColumn("fraud_score",
            F.col("is_large_amount").cast("int") +
            F.col("is_suspicious_ip").cast("int") +
            F.col("is_unknown_merchant").cast("int") +
            F.col("is_pending_status").cast("int") +
            F.col("is_unusual_currency").cast("int"))
        .withColumn("risk_level",
            F.when(F.col("fraud_score") >= 4, F.lit("high"))
             .when(F.col("fraud_score") >= 2, F.lit("medium"))
             .otherwise(F.lit("low")))
    )

print("add_fraud_signals defined — 5 signals")

# ─────────────────────────────────────────────────────────────
# STAGE 4 — CURRENCY NORMALISATION
# ─────────────────────────────────────────────────────────────
def add_eur_amount(df):
    eur_expr = F.when(F.col("currency") == "EUR", F.col("amount"))
    for currency, rate in CONFIG["fx_rates_to_eur"].items():
        if currency != "EUR":
            eur_expr = eur_expr.when(
                F.col("currency") == currency,
                F.round(F.col("amount") * F.lit(rate), 2)
            )
    eur_expr = eur_expr.otherwise(F.col("amount"))
    return df.withColumn(
        "amount_eur",
        eur_expr.cast(DecimalType(15, 2))
    )

print("add_eur_amount defined")

# ─────────────────────────────────────────────────────────────
# STAGE 5 — SILVER METADATA
# ─────────────────────────────────────────────────────────────
def add_silver_metadata(df):
    return (df
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id",       F.lit(RUN_ID))
    )

print("add_silver_metadata defined")
print()
print("All Silver functions ready — run Cell 6")

add_validation defined — 7 rules
cast_types defined
add_fraud_signals defined — 5 signals
add_eur_amount defined
add_silver_metadata defined

All Silver functions ready — run Cell 6


In [5]:
# ─────────────────────────────────────────────────────────────
# Cell 6 — Fixed — recreate bronze_stream before starting
#
# The bronze_stream variable from Cell 4 may be stale
# if the Spark session was restarted
# We recreate it here to be safe before calling writeStream
# ─────────────────────────────────────────────────────────────

from delta.tables import DeltaTable
from pymongo import MongoClient
from pyspark.sql import functions as F
from datetime import datetime
import traceback

# ─────────────────────────────────────────────────────────────
# Recreate Bronze stream source
# Safe to call even if Cell 4 already ran
# getOrCreate on the Spark session handles duplicates
# ─────────────────────────────────────────────────────────────
BRONZE_DELTA_PATH = f"s3a://{CONFIG['bronze_bucket']}/delta/transactions/"
SILVER_DELTA_PATH = f"s3a://{CONFIG['silver_bucket']}/delta/transactions/"
QUARANTINE_PATH   = f"s3a://{CONFIG['quarantine_bucket']}/silver/transactions/"
CHECKPOINT_PATH   = f"s3a://{CONFIG['silver_bucket']}/checkpoints/silver-stream/"

# Recreate bronze stream — safe to call multiple times
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", "10")
    .option("ignoreDeletes",      "true")
    .option("ignoreChanges",      "true")
    .load(BRONZE_DELTA_PATH)
)

print("Bronze stream source recreated")
print(f"Reading from:  {BRONZE_DELTA_PATH}")
print(f"Writing to:    {SILVER_DELTA_PATH}")
print(f"Quarantine:    {QUARANTINE_PATH}")
print(f"Checkpoint:    {CHECKPOINT_PATH}")

# ─────────────────────────────────────────────────────────────
# BATCH PROCESSING FUNCTION
# ─────────────────────────────────────────────────────────────
def process_silver_batch(batch_df, batch_id):
    record_count = batch_df.count()

    if record_count == 0:
        print(f"Batch {batch_id}: empty — skipping")
        return

    print(f"Batch {batch_id}: processing {record_count} records...")

    try:
        # Validate
        validated_df = add_validation(batch_df)
        valid_df     = validated_df.filter(
            F.col("validation_error").isNull()
        ).drop("validation_error")
        quarantine_df = validated_df.filter(
            F.col("validation_error").isNotNull()
        )

        # Transform valid records
        silver_df = cast_types(valid_df)
        silver_df = add_fraud_signals(silver_df)
        silver_df = add_eur_amount(silver_df)
        silver_df = add_silver_metadata(silver_df)

        valid_count      = silver_df.count()
        quarantine_count = quarantine_df.count()

        print(f"  Valid:      {valid_count} → Silver")
        print(f"  Quarantine: {quarantine_count} → Quarantine")

    except Exception as e:
        print(f"Batch {batch_id} transformation failed: {e}")
        traceback.print_exc()
        return

    # Write valid to Silver Delta Lake
    try:
        if valid_count > 0:
            (silver_df
                .write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .partitionBy("processing_date", "bank_code")
                .save(SILVER_DELTA_PATH)
            )
            print(f"  Written {valid_count} to Silver")
    except Exception as e:
        print(f"  Silver write failed: {e}")
        traceback.print_exc()

    # Write invalid to quarantine
    try:
        if quarantine_count > 0:
            (quarantine_df
                .withColumn("quarantine_reason",
                    F.col("validation_error"))
                .withColumn("quarantined_at",
                    F.current_timestamp())
                .withColumn("silver_run_id", F.lit(RUN_ID))
                .withColumn("batch_id",      F.lit(batch_id))
                .drop("validation_error")
                .write
                .format("json")
                .mode("append")
                .save(QUARANTINE_PATH)
            )
            print(f"  Written {quarantine_count} to quarantine")
    except Exception as e:
        print(f"  Quarantine write failed: {e}")

    # Log to MongoDB
    try:
        client = MongoClient(CONFIG["mongodb_uri"])
        db     = client[CONFIG["mongodb_db"]]
        db["silver_batches"].insert_one({
            "run_id":             RUN_ID,
            "batch_id":           batch_id,
            "pipeline_name":      CONFIG["pipeline_name"],
            "processing_date":    CONFIG["processing_date"],
            "records_received":   record_count,
            "records_valid":      valid_count,
            "records_quarantine": quarantine_count,
            "processed_at":       datetime.utcnow().isoformat(),
            "status":             "SUCCESS"
        })
        client.close()
        print(f"  Logged to MongoDB")
    except Exception as e:
        print(f"  MongoDB logging failed: {e}")

    print(f"Batch {batch_id}: complete")
    print()

# ─────────────────────────────────────────────────────────────
# START SILVER STREAM
# ─────────────────────────────────────────────────────────────
print()
print("Starting Silver stream...")

silver_query = (
    bronze_stream
    .writeStream
    .foreachBatch(process_silver_batch)
    .outputMode("update")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=CONFIG["trigger_interval"])
    .start()
)

print(f"Silver stream started")
print(f"Query ID:      {silver_query.id}")
print(f"Stream active: {silver_query.isActive}")
print()
print("Waiting for first batch — runs every 30 seconds...")

Bronze stream source recreated
Reading from:  s3a://bronze/delta/transactions/
Writing to:    s3a://silver/delta/transactions/
Quarantine:    s3a://quarantine/silver/transactions/
Checkpoint:    s3a://silver/checkpoints/silver-stream/

Starting Silver stream...
Silver stream started
Query ID:      7e154bf8-b409-4cf0-aaa3-299f79cd3d01
Stream active: True

Waiting for first batch — runs every 30 seconds...


In [6]:
# ─────────────────────────────────────────────────────────────
# Cell 7 — Inspect Silver Delta Lake
#
# Check what landed in Silver
# Verify fraud signals are working
# Check quarantine is empty (expected with clean simulator data)
# Inspect the full Silver schema
# ─────────────────────────────────────────────────────────────

import time
from pyspark.sql import functions as F

print("=" * 55)
print("SILVER STREAM STATUS")
print("=" * 55)
print(f"Stream active:  {silver_query.isActive}")
print(f"Exception:      {silver_query.exception()}")

progress = silver_query.lastProgress
if progress:
    print(f"Last batch ID:  {progress['batchId']}")
    print(f"Input rows:     {progress['numInputRows']}")

# ─────────────────────────────────────────────────────────────
# READ SILVER DELTA LAKE AS BATCH
# ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("SILVER DELTA LAKE")
print("=" * 55)

try:
    silver_df = spark.read.format("delta").load(SILVER_DELTA_PATH)
    total     = silver_df.count()
    print(f"Total records in Silver: {total}")
    print()

    # Records per bank
    print("Records per bank:")
    silver_df.groupBy("bank_code") \
        .agg(F.count("*").alias("count")) \
        .orderBy("bank_code") \
        .show()

    # Fraud signal breakdown
    print("Fraud score distribution:")
    silver_df.groupBy("fraud_score", "risk_level") \
        .agg(F.count("*").alias("count")) \
        .orderBy("fraud_score") \
        .show()

    # High risk transactions
    high_risk = silver_df.filter(F.col("risk_level") == "high")
    print(f"High risk transactions: {high_risk.count()}")
    if high_risk.count() > 0:
        print()
        print("High risk sample:")
        high_risk.select(
            "transaction_id",
            "bank_code",
            "amount",
            "amount_eur",
            "currency",
            "status",
            "is_large_amount",
            "is_suspicious_ip",
            "is_unknown_merchant",
            "fraud_score",
            "risk_level"
        ).show(5, truncate=False)

    # Amount comparison — original vs EUR
    print("Average amounts by bank:")
    silver_df.groupBy("bank_code") \
        .agg(
            F.round(F.avg("amount"),     2).alias("avg_amount_original"),
            F.round(F.avg("amount_eur"), 2).alias("avg_amount_eur")
        ) \
        .orderBy("bank_code") \
        .show()

    # Verify types are correct
    print("Silver schema — verify proper types:")
    silver_df.select(
        "transaction_id",
        "amount",
        "amount_eur",
        "event_timestamp",
        "fraud_score",
        "risk_level",
        "silver_processed_at"
    ).printSchema()

except Exception as e:
    print(f"Error reading Silver: {e}")

# ─────────────────────────────────────────────────────────────
# CHECK QUARANTINE
# ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("QUARANTINE BUCKET")
print("=" * 55)

try:
    quarantine_df = spark.read.format("json").load(QUARANTINE_PATH)
    q_count       = quarantine_df.count()
    print(f"Quarantined records: {q_count}")
    if q_count > 0:
        print()
        print("Quarantine reasons:")
        quarantine_df.groupBy("quarantine_reason") \
            .agg(F.count("*").alias("count")) \
            .orderBy("count", ascending=False) \
            .show()
except Exception as e:
    print(f"Quarantine empty or not yet created: {e}")

# ─────────────────────────────────────────────────────────────
# CHECK MONGODB AUDIT LOG
# ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("MONGODB AUDIT LOG")
print("=" * 55)

try:
    from pymongo import MongoClient
    client   = MongoClient(CONFIG["mongodb_uri"],
                           serverSelectionTimeoutMS=3000)
    db       = client[CONFIG["mongodb_db"]]
    col      = db["silver_batches"]
    total_batches = col.count_documents({})
    print(f"Total batches logged: {total_batches}")

    # Get last 3 batches
    print()
    print("Last 3 batches:")
    for doc in col.find().sort("batch_id", -1).limit(3):
        print(f"  Batch {doc['batch_id']:3d} — "
              f"received: {doc['records_received']:4d} — "
              f"valid: {doc['records_valid']:4d} — "
              f"quarantine: {doc['records_quarantine']:3d} — "
              f"{doc['processed_at']}")
    client.close()
except Exception as e:
    print(f"MongoDB error: {e}")

SILVER STREAM STATUS
Stream active:  True
Exception:      None

SILVER DELTA LAKE
Total records in Silver: 569

Records per bank:
+---------+-----+
|bank_code|count|
+---------+-----+
|      ING|  252|
|    MBANK|  126|
|      PKO|  191|
+---------+-----+

Fraud score distribution:
+-----------+----------+-----+
|fraud_score|risk_level|count|
+-----------+----------+-----+
|          0|       low|  529|
|          4|      high|   40|
+-----------+----------+-----+

High risk transactions: 40

High risk sample:
+------------------------+---------+--------+----------+--------+-------+---------------+----------------+-------------------+-----------+----------+
|transaction_id          |bank_code|amount  |amount_eur|currency|status |is_large_amount|is_suspicious_ip|is_unknown_merchant|fraud_score|risk_level|
+------------------------+---------+--------+----------+--------+-------+---------------+----------------+-------------------+-----------+----------+
|TXN-ING-FRAUD-D5F13CED  |ING     